In [ ]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

# SQL query

In [ ]:
auth.authenticate_user()

client = bigquery.Client(project="data-analytics-mate")

sql_query = """
with session_info as(


  select
        s.date,
        s.ga_session_id,
        sp.country,
        sp.device,
        sp.continent,
        sp.channel,
        ab.test,
        ab.test_group
  from `DA.ab_test` ab
  join `DA.session` s
  on ab.ga_session_id = s.ga_session_id
  join `DA.session_params` sp
  on sp.ga_session_id = ab.ga_session_id
),


session_with_orders as(
select
        session_info.date,
        session_info.ga_session_id,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        count(distinct o.ga_session_id) as session_with_orders
from `DA.order` o
join session_info
on o.ga_session_id = session_info.ga_session_id
group by
        session_info.date,
        session_info.ga_session_id,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
),


events as (select
        session_info.date,
        session_info.ga_session_id,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        sp.event_name,
        count(sp.ga_session_id) as event_cnt
from `DA.event_params` sp
join session_info
on sp.ga_session_id = session_info.ga_session_id
group by
        session_info.date,
        session_info.ga_session_id,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        sp.event_name
),


session as(
select
        session_info.date,
        session_info.ga_session_id,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        count(distinct session_info.ga_session_id) as session_cnt
from session_info
group by
        session_info.date,
        session_info.ga_session_id,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
),


account as(
select
        session_info.date,
        session_info.ga_session_id,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        count(distinct acs.ga_session_id) as new_account_cnt
from `DA.account_session` acs
join session_info
on acs.ga_session_id = session_info.ga_session_id
group by
        session_info.date,
        session_info.ga_session_id,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
)


select
        session_with_orders.date,
        session_with_orders.ga_session_id,
        session_with_orders.country,
        session_with_orders.device,
        session_with_orders.continent,
        session_with_orders.channel,
        session_with_orders.test,
        session_with_orders.test_group,
        'session_with_orders' as event_name,
        session_with_orders.session_with_orders as value
from session_with_orders
union all
select
        events.date,
        events.ga_session_id,
        events.country,
        events.device,
        events.continent,
        events.channel,
        events.test,
        events.test_group,
        event_name,
        event_cnt as value
from events
union all
select
        session.date,
        session.ga_session_id,
        session.country,
        session.device,
        session.continent,
        session.channel,
        session.test,
        session.test_group,
        'session' as event_name,
        session_cnt as value
from session
union all
select
        account.date,
        account.ga_session_id,
        account.country,
        account.device,
        account.continent,
        account.channel,
        account.test,
        account.test_group,
        'new_accounts' as event_name,
        new_account_cnt as value
from account
"""
df = client.query(sql_query).to_dataframe(create_bqstorage_client=False)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214136 entries, 0 to 3214135
Data columns (total 10 columns):
 #   Column         Dtype 
---  ------         ----- 
 0   date           dbdate
 1   ga_session_id  Int64 
 2   country        object
 3   device         object
 4   continent      object
 5   channel        object
 6   test           Int64 
 7   test_group     Int64 
 8   event_name     object
 9   value          Int64 
dtypes: Int64(4), dbdate(1), object(5)
memory usage: 257.5+ MB


# Calculation of A/B testing

In [ ]:
# Checking event names

print(df['event_name'].unique())

['new_accounts' 'session_with_orders' 'session' 'page_view' 'scroll'
 'add_shipping_info' 'session_start' 'view_item' 'user_engagement'
 'add_to_cart' 'first_visit' 'view_promotion' 'view_search_results'
 'add_payment_info' 'select_item' 'begin_checkout' 'select_promotion'
 'click' 'view_item_list']


In [ ]:
# Creating a list of metrics
metrics = [
    {"name": "add_payment_info / session", "num": "add_payment_info", "den": "session"},
    {"name": "add_shipping_info / session", "num": "add_shipping_info", "den": "session"},
    {"name": "begin_checkout / session", "num": "begin_checkout", "den": "session"},
    {"name": "new_accounts / session", "num": "new_accounts", "den": "session"},
    {"name": "add_to_cart / session", "num": "add_to_cart", "den": "session"}
]

results = []

for test_num in df['test'].unique():
    test_data = df[df['test'] == test_num]
    for m in metrics:

        control = test_data[test_data['test_group'] == 1]
        num_control = control[control['event_name'] == m['num']]['value'].sum()
        den_control = control[control['event_name'] == m['den']]['value'].sum()

        variant = test_data[test_data['test_group'] == 2]
        num_variant = variant[variant['event_name'] == m['num']]['value'].sum()
        den_variant = variant[variant['event_name'] == m['den']]['value'].sum()

       # Division by zero check
        if den_control == 0 or den_variant == 0:
            continue

        # How many events occurred for Z test
        counts = np.array([num_variant, num_control])
        #  How many sessions occurred for Z test
        nobs = np.array([den_variant, den_control])

        z_stat, p_value = proportions_ztest(counts, nobs)

        results.append({
            "test_number": test_num,
            "metric": m['name'],
            "conversion_rate_control": num_control / den_control,
            "conversion_rate_test": num_variant / den_variant,
            "p_value": p_value,
            "z_stat": z_stat,
            "significant": p_value < 0.05
        })

final_df = pd.DataFrame(results)

final_df.head()

,test_number,metric,conversion_rate_control,conversion_rate_test,p_value,z_stat,significant
0,2,add_payment_info / session,0.046290,0.047946,0.214608,1.240994,False
1,2,add_shipping_info / session,0.068724,0.069859,0.477979,0.709557,False
2,2,begin_checkout / session,0.084168,0.085841,0.340642,0.952898,False
3,2,new_accounts / session,0.082252,0.083274,0.556000,0.588793,False
4,2,add_to_cart / session,0.055513,0.060923,0.000243,3.669417,True


In [ ]:
from google.colab import files

final_df.to_csv('ab_test_results.csv', index=False)
files.download('ab_test_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# General conclusions

Для оцінки результатів A/B тесту використано Z-тест для двох часток. Задача виявити статистично значущі зміни в метриках та відкинути випадковості

**Тест 1**

Ми маємо статистично значуще зростання одразу кількох ключових метрик
- Конверсія begin_checkout виросла з 8,3% до 8,8%, p-value 0.002
- add_payment_info виросло з 4,3% до 4,9%, p-value 0.00008
- add_shipping_info виросло з 6,6% до 7,1%, p-value 0.009


**Тест 2**

Метрика add_to_cart показала значуще зростання з 5,5% до 6,0% з p-value 0.0002
Проте, це не допомогло конвертуватись в продажі, бо на етапах чекауту та оплат значущих змін немає всі p-value більше 0.05. Користувачі просто додають товари в кошик, але далі не йдуть



**Тест 3**

Метрика begin_checkout статистично значущо впала з 13,6% до 13,1% з p-value 0.012. А також значущо просіла метрика add_to_cart з 25,2% до 24,4%, p-value 0.0008. Тож бачимо, що зміни негативно впливають на оформлення замовлення

**Тест 4**

Значущо впали реєстрації new_accounts з 8,5% до 8,2%, p-value = 0.017.
Переходи до begin_checkout також значущо знизились з 11,9% до 11,6%, p-value 0.045. Тож гіпотеза провалилась



**Висновки**
- Зміни з Тесту 1 масштабувати на всіх користувачів, оскільки вони доведено покращують фінальні кроки перед покупкою
- Гіпотези з Тестів 3 та 4 негативно впливають на повеінку користувачів для конверсії, тож потрібно припинити їх використання
- Зміни з Тесту 2 потрібно дослідити. Бо вони стимулюють наповнювати кошик, але потрібно зрозуміти, чому ці кошики кидають

[Дашборд з розрахованими метриками в Tableau Public](https://public.tableau.com/views/A_BTestingTool_17781304401450/CalculationMetrics?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)

[CSV файл](https://drive.google.com/file/d/1n3xY4Ygqx2DRdKgtqt7Ewme41Uw7aWAe/view?usp=drive_link)


[Дашборд з розподіленням вибірки в Tableau Public](https://public.tableau.com/views/ABTestingTool_17762612381770/ABTest?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)